# 식별자 연결 복구 연습

두 표를 붙인 뒤 **같은 기록이 늘어났거나**, **구역이 빈 기록의 이름을 찾지 못했을 때** 이 연습을 사용하자. 한 행의 기록 이름은 “어느 관찰인가”, 그 행의 위치 코드는 “어느 위치표 행에서 정보를 가져올까”에 답한다. 아래 자료는 이 노트북 안에 모두 들어 있다.

### 겹친 이름부터 읽기

점검표의 한 줄은 방문 한 번이다. 위치표는 위치 이름마다 구역 하나를 알려 줘야 한다. 아래에서 `B1`을 찾는 방문에 두 구역이 붙을 수 있는 이유를 먼저 찾아보자. 양쪽의 빈 이름도 살피자. 이름을 모르는 두 대상을 같은 위치라고 정할 근거가 있을까?

In [ ]:
import pandas as pd

visits = pd.DataFrame({
    "ticket": ["M14", "M07", "M21", "M12"],
    "place": [" C2 ", "", "B1", "X9"],
})
location_list = pd.DataFrame({
    "code": ["B1", " C2", "B1", ""],
    "area": ["West", "East", "North", "Top"],
})
print(visits.to_string(index=False))
print(location_list.to_string(index=False))

### 한 번에 한 단계씩 확인하기

① 두 파일의 **이름 열만** 문자로 맞추고 앞뒤 공백을 뗀다. 빈 문자열은 결측으로 표시한다. ② 위치표에서 모르는 이름을 제외한 뒤 이름 하나에 여러 행이 달리는지 본다. `duplicated(keep=False)`는 중복된 모든 행을 표시한다. 상충하는 구역은 원본을 확인해야 정할 수 있다.

In [ ]:
def text_id(values):
    return values.astype("string").str.strip().replace("", pd.NA)

left = visits.assign(place=text_id(visits["place"]))
right = location_list.assign(code=text_id(location_list["code"]))
known = right.loc[right["code"].notna()]
print(known.loc[known["code"].duplicated(keep=False)].to_string(index=False))

③ 시설 담당자가 원본에서 B1은 West, C2는 East라고 확인했다. 확인된 위치표를 왼쪽 방문에 붙이고, 연결되지 않은 행에서 **방문 이름**을 읽는다. M07은 위치가 비었고 M12의 X9는 위치표에 없다. 둘 다 원래 관찰로 남는다.

In [ ]:
verified = pd.DataFrame({"code": ["B1", "C2"], "area": ["West", "East"]})
paired = left.merge(verified, left_on="place", right_on="code", how="left",
                    validate="many_to_one", indicator=True)
unlinked = paired.loc[paired["_merge"] == "left_only", "ticket"].tolist()
print(paired[["ticket", "place", "area", "_merge"]].to_string(index=False))
print("연결 못 한 기록", unlinked)
assert len(paired) == len(visits) and unlinked == ["M07", "M12"]

### 도움 없이 새 기록으로 확인하기

아래 새 자료에서 중복된 위치 이름과 영향을 받는 방문 이름을 먼저 적자. 시설 원본은 G1이 North, G2가 South라고 확인해 주었다. 그 근거로 위치표를 바로잡고, 빈 이름끼리 결합되지 않았는지 **실제 미연결 방문 이름과 남은 행 수**로 확인하자. 위 풀이를 다시 봤다면 어느 줄이었는지 적자.

첫 예상 · 도움 여부 · 확인 결과 →

이 기록을 마치면 [본 교안의 ‘새 표의 연결을 다시 확인하기’](week01_foundations.ipynb#merge-recheck)로 돌아가 다른 반납함의 기록으로 자기 절차를 다시 시험하자.

In [ ]:
fresh_visits = pd.DataFrame({
    "ticket": ["J6", "J2", "J9"], "place": ["G1", " ", " G2"],
})
fresh_places = pd.DataFrame({
    "code": ["G2", "", "G1", "G2"], "area": ["West", "Top", "North", "South"],
})
# 고친 표, 실제 미연결 이름, 남은 행 수를 남기자.